# losslens: a tour

**losslens** shows what the loss looks like around a trained neural network.

A network is a point in a space with millions of dimensions, one per weight. We can't look at that
space directly, so losslens picks **two random directions** in it, moves the trained weights a
little along each, and measures the loss at every point of a grid. The result is a map of the loss
around the trained model:

- a **wide, flat valley** means the loss barely changes when the weights change a bit;
- a **narrow, sharp valley** means small changes to the weights quickly hurt the model.

This notebook goes through everything the tool offers:

1. Setup
2. Train a model (the part students already did in their homework)
3. Compute a landscape
4. The plots
5. The numbers: `summary()`
6. The options: `scale`, `resolution`, `n_samples`, `seed`
7. Comparing models
8. Other kinds of problems: regression, autoencoders, custom batches
9. Saving and loading

Run it on a GPU runtime (in Colab: *Runtime → Change runtime type → GPU*).

## 1. Setup

In [ ]:
%pip install -q git+https://github.com/iturbideM/losslens

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as T
from torch.utils.data import DataLoader
from IPython.display import display

import losslens

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)
torch.manual_seed(0)

## 2. Train a model

This section is *not* part of losslens: it stands in for the model a student trained in an earlier
homework. We train two small ResNets on CIFAR-10, identical except that one has **skip
connections** and the other doesn't. We'll compare their landscapes in section 7.

Training takes a few minutes on a GPU.

In [ ]:
EPOCHS = 5

normalize = T.Normalize((0.491, 0.482, 0.447), (0.247, 0.243, 0.262))
train_tf = T.Compose([T.RandomCrop(32, padding=4), T.RandomHorizontalFlip(), T.ToTensor(), normalize])
plain_tf = T.Compose([T.ToTensor(), normalize])

train_set = torchvision.datasets.CIFAR10("data", train=True, download=True, transform=train_tf)
train_loader = DataLoader(train_set, batch_size=128, shuffle=True, num_workers=2)

# Same images, without augmentation: this is what we give to losslens.
eval_set = torchvision.datasets.CIFAR10("data", train=True, download=True, transform=plain_tf)
eval_loader = DataLoader(eval_set, batch_size=256)

In [ ]:
class Block(nn.Module):
    def __init__(self, cin, cout, stride, skip):
        super().__init__()
        self.skip = skip
        self.conv1 = nn.Conv2d(cin, cout, 3, stride, 1, bias=False)
        self.bn1 = nn.BatchNorm2d(cout)
        self.conv2 = nn.Conv2d(cout, cout, 3, 1, 1, bias=False)
        self.bn2 = nn.BatchNorm2d(cout)
        self.shortcut = nn.Sequential()
        if stride != 1 or cin != cout:
            self.shortcut = nn.Sequential(nn.Conv2d(cin, cout, 1, stride, bias=False), nn.BatchNorm2d(cout))

    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        if self.skip:
            out = out + self.shortcut(x)
        return F.relu(out)


class ResNet(nn.Module):
    def __init__(self, depth=20, skip=True):
        super().__init__()
        n = (depth - 2) // 6
        self.stem = nn.Sequential(nn.Conv2d(3, 16, 3, 1, 1, bias=False), nn.BatchNorm2d(16), nn.ReLU())
        layers, cin = [], 16
        for cout, stride in [(16, 1), (32, 2), (64, 2)]:
            for i in range(n):
                layers.append(Block(cin, cout, stride if i == 0 else 1, skip))
                cin = cout
        self.layers = nn.Sequential(*layers)
        self.fc = nn.Linear(64, 10)

    def forward(self, x):
        x = self.layers(self.stem(x))
        return self.fc(F.adaptive_avg_pool2d(x, 1).flatten(1))


def train(model, epochs=EPOCHS):
    model.to(device)
    opt = torch.optim.SGD(model.parameters(), lr=0.1, momentum=0.9, weight_decay=5e-4)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, 0.1, total_steps=epochs * len(train_loader))
    for epoch in range(epochs):
        model.train()
        correct = total = 0
        for x, y in train_loader:
            x, y = x.to(device), y.to(device)
            out = model(x)
            loss = F.cross_entropy(out, y)
            opt.zero_grad(); loss.backward(); opt.step(); sched.step()
            correct += (out.argmax(1) == y).sum().item(); total += len(y)
        print(f"epoch {epoch + 1}: train accuracy {100 * correct / total:.1f}%")
    return model

In [ ]:
resnet = train(ResNet(depth=20, skip=True))

## 3. Compute a landscape

One call. losslens needs three things:

- the **trained model**,
- some **data** (a DataLoader, a Dataset, or a tuple `(X, y)` of tensors),
- the **loss function**, called as `loss_fn(model(inputs), targets)`.

Use data **without augmentation**, so the loss reflects the model and not the random crops.

Before starting, losslens prints how many points it will evaluate and an estimate of the time.

In [ ]:
surf = losslens.surface(resnet, eval_loader, loss_fn=nn.CrossEntropyLoss())
surf

The student's model is not modified: losslens works on a copy. Biases, BatchNorm parameters and
frozen layers are never moved, and the loss is computed in eval mode (dropout off).

## 4. The plots

### Contour map

The **red star** is the trained model, at the center. Each axis is one random direction.
Darker means lower loss.

In [ ]:
surf.plot()

### 3D surface

The same data as a surface. It's interactive: drag to rotate, scroll to zoom, hover to read values.

In [ ]:
surf.plot_3d()

### Slices

The two 1D cuts through the trained model, one along each direction. Often the easiest way to see
how steep the valley is.

In [ ]:
surf.plot_slice()

### Color scale: `log` and `zmax`

By default the top of the color scale is set so that a few huge values far from the model don't
wash out the rest. Two arguments change that, on every plot function:

- `log=True` shows log10 of the loss, useful when it grows by orders of magnitude;
- `zmax=` sets the top of the color scale by hand.

In [ ]:
display(surf.plot(log=True, title="log scale"))
display(surf.plot(zmax=3, title="zmax=3"))

## 5. The numbers: `summary()`

Pictures are nice, but a report needs numbers. `summary()` prints a table and returns the same values
as a dictionary.

- **Loss of the trained model**, and the **lowest loss on the grid** and where it is. If the lowest
  point isn't the trained model, the model is not at a minimum along these directions.
- **Mean loss increase at 10%, 25% and 50% of scale**: the average rise in loss on a circle at that
  distance from the model. Smaller means flatter.
- **Steeper / flatter direction**: 1.0 means the valley looks the same in both directions.

In [ ]:
numbers = surf.summary()

In [ ]:
numbers["rise_at_25%"]

## 6. The options

`surface()` has only a few options, and all of them are about *what to look at*. The method itself
is fixed, so all plots are made the same way and are comparable.

| Option | Default | What it does |
|---|---|---|
| `scale` | `1.0` | How far from the model to look. At 1.0, each layer's weights change by up to 100% of their own size. |
| `resolution` | `25` | Grid points per side: sharper but slower (25 means 625 loss evaluations). |
| `n_samples` | `2000` | How many examples the loss is averaged over. |
| `seed` | `0` | Which random directions (and which examples). |

### `scale`: how far to look

Too small and you only see the bottom of the valley; too large and the valley is a tiny dot in a
plateau of very high loss.

In [ ]:
for s in [0.25, 1.0, 2.0]:
    display(losslens.surface(resnet, eval_loader, loss_fn=nn.CrossEntropyLoss(), scale=s)
            .plot(title=f"scale={s}"))

### `resolution`: how sharp

The time grows with the square of the resolution, so try a low value first and increase it for the
final figure.

In [ ]:
coarse = losslens.surface(resnet, eval_loader, loss_fn=nn.CrossEntropyLoss(), resolution=11)
coarse.plot(title="resolution=11")

### `n_samples`: how much data

More examples give a smoother, more accurate surface, and take longer. 2000 is usually enough to see
the shape.

In [ ]:
small = losslens.surface(resnet, eval_loader, loss_fn=nn.CrossEntropyLoss(), n_samples=200)
small.plot(title="n_samples=200")

### `seed`: is the shape real?

The directions are random, so a good habit is to check the picture with a second seed. If both look
alike, the shape is a property of the model and not of one lucky direction.

In [ ]:
other_seed = losslens.surface(resnet, eval_loader, loss_fn=nn.CrossEntropyLoss(), seed=1)
losslens.compare({"seed 0": surf, "seed 1": other_seed})

## 7. Comparing models

`compare()` draws several landscapes side by side on **one shared color scale** and prints their
numbers in one table.

This is fair even between different models: the directions are scaled to each layer's own weights,
so "1.0" means "change each layer by its own size" for every model. Just use the same `scale` and
`n_samples` (compare warns you if you don't).

Here: the same ResNet-20, with and without skip connections.

In [ ]:
plain = train(ResNet(depth=20, skip=False))

In [ ]:
surf_plain = losslens.surface(plain, eval_loader, loss_fn=nn.CrossEntropyLoss())
losslens.compare({"with skip connections": surf, "without": surf_plain})

What do you see? The original paper found that without skip connections, landscapes become sharper
and more chaotic as networks get deeper. At 20 layers the difference may be modest; try
`depth=56` to make it stronger (training takes longer).

## 8. Other kinds of problems

### Regression, with plain tensors

Any loss works, and data can be a tuple of tensors.

In [ ]:
X = torch.linspace(-3, 3, 1000).unsqueeze(1)
y = torch.sin(2 * X) + 0.1 * torch.randn_like(X)

mlp = nn.Sequential(nn.Linear(1, 64), nn.Tanh(), nn.Linear(64, 64), nn.Tanh(), nn.Linear(64, 1))
opt = torch.optim.Adam(mlp.parameters(), lr=1e-2)
for step in range(2000):
    opt.zero_grad(); F.mse_loss(mlp(X), y).backward(); opt.step()

surf_reg = losslens.surface(mlp, (X, y), loss_fn=nn.MSELoss())
surf_reg.plot(title="regression MLP")

### Anything else: `forward=`

If the batches aren't `(inputs, targets)`, or the loss isn't `loss_fn(model(inputs), targets)`,
pass `forward(model, batch)` instead: a function that returns the loss for one batch.

An autoencoder, for example, compares its output with its own input and ignores the labels:

In [ ]:
autoencoder = nn.Sequential(nn.Flatten(), nn.Linear(3072, 128), nn.ReLU(), nn.Linear(128, 3072)).to(device)
opt = torch.optim.Adam(autoencoder.parameters(), lr=1e-3)
for x, _ in train_loader:
    x = x.to(device)
    opt.zero_grad(); F.mse_loss(autoencoder(x), x.flatten(1)).backward(); opt.step()

surf_ae = losslens.surface(
    autoencoder, eval_loader,
    forward=lambda model, batch: F.mse_loss(model(batch[0]), batch[0].flatten(1)),
)
surf_ae.plot(title="autoencoder")

The same idea covers models that take dictionaries (for example HuggingFace-style batches):

```python
surf = losslens.surface(net, loader, forward=lambda model, batch: model(**batch).loss)
```

## 9. Saving and loading

Computing a landscape can take minutes. Save it, and load it later to re-plot without recomputing.

In [ ]:
surf.save("resnet20.npz")
again = losslens.load("resnet20.npz")
again.plot(title="loaded from file")

## Summary

```python
surf = losslens.surface(model, data, loss_fn=...)   # or forward=...
surf.plot()   surf.plot_3d()   surf.plot_slice()     # log=True, zmax=...
surf.summary()
losslens.compare({"A": surf_a, "B": surf_b})
surf.save("a.npz");  losslens.load("a.npz")
```

Method: Li et al., *Visualizing the Loss Landscape of Neural Nets*, NeurIPS 2018.